# Coinbase trades and Level 2 ticks

This notebook opens Coinbase Advanced Trade's **public** WebSocket, subscribes to `market_trades` and `level2` for `SOL-USD`, and saves a short sample of each. No API key is required.

Coinbase groups trades into messages over roughly 250 ms. Each item inside `trades` is still an individual trade. The reported `side` is the **maker side**, so it is stored below as `maker_side`. Level 2 begins with a book snapshot and then sends updates; `new_quantity` replaces the complete quantity at that price level, and zero removes the level. `heartbeats` keeps a quiet connection open.

In [1]:
%pip install -q "websockets>=14" pandas


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import asyncio
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from websockets.asyncio.client import connect

WS_URL = "wss://advanced-trade-ws.coinbase.com"
PRODUCT_IDS = ["SOL-USD"]
CAPTURE_SECONDS = 3 * 60 * 60

async def collect_market_ticks(product_ids, seconds):
    trade_rows = []
    level2_rows = []
    loop = asyncio.get_running_loop()
    deadline = loop.time() + seconds

    async with connect(WS_URL, max_size=None) as socket:
        await socket.send(json.dumps({
            "type": "subscribe",
            "channel": "market_trades",
            "product_ids": product_ids,
        }))
        await socket.send(json.dumps({
            "type": "subscribe",
            "channel": "level2",
            "product_ids": product_ids,
        }))
        await socket.send(json.dumps({
            "type": "subscribe",
            "channel": "heartbeats",
        }))

        while (remaining := deadline - loop.time()) > 0:
            try:
                message = json.loads(
                    await asyncio.wait_for(socket.recv(), timeout=remaining)
                )
            except asyncio.TimeoutError:
                break

            received_at_ns = time.time_ns()
            if message.get("channel") == "market_trades":
                for event in message.get("events", []):
                    for trade in event.get("trades", []):
                        trade_rows.append({
                            "exchange": "coinbase",
                            "product_id": trade["product_id"],
                            "trade_id": trade["trade_id"],
                            "exchange_time": trade["time"],
                            "received_at_ns": received_at_ns,
                            "price": trade["price"],
                            "size": trade["size"],
                            "maker_side": trade["side"],
                            "event_type": event.get("type"),
                            "sequence_num": message.get("sequence_num"),
                        })

            elif message.get("channel") == "l2_data":
                for event in message.get("events", []):
                    for update in event.get("updates", []):
                        level2_rows.append({
                            "exchange": "coinbase",
                            "product_id": event["product_id"],
                            "event_type": event.get("type"),
                            "event_time": update.get("event_time"),
                            "message_time": message.get("timestamp"),
                            "received_at_ns": received_at_ns,
                            "side": update["side"],
                            "price_level": update["price_level"],
                            "new_quantity": update["new_quantity"],
                            "sequence_num": message.get("sequence_num"),
                        })
    return trade_rows, level2_rows

trade_rows, level2_rows = await collect_market_ticks(PRODUCT_IDS, CAPTURE_SECONDS)
trades = pd.DataFrame(trade_rows)
level2 = pd.DataFrame(level2_rows)
print(f"Collected {len(trades):,} trades and {len(level2):,} Level 2 rows")
display(trades.head())
display(level2.head())

Collected 15,106 trades and 942,749 Level 2 rows


,exchange,product_id,trade_id,exchange_time,received_at_ns,price,size,maker_side,event_type,sequence_num
0,coinbase,SOL-USD,357178420,2026-10-09T10:38:45.358641Z,1791542326478871000,109.6,14.00451408,BUY,snapshot,0
1,coinbase,SOL-USD,357178419,2026-10-09T10:38:45.358641Z,1791542326478871000,109.61,0.00078272,BUY,snapshot,0
2,coinbase,SOL-USD,357178418,2026-10-09T10:38:45.358641Z,1791542326478871000,109.61,0.00012936,BUY,snapshot,0
3,coinbase,SOL-USD,357178417,2026-10-09T10:38:45.358641Z,1791542326478871000,109.62,0.0000531,BUY,snapshot,0
4,coinbase,SOL-USD,357178416,2026-10-09T10:38:45.231261Z,1791542326478871000,109.62,0.0109469,BUY,snapshot,0


,exchange,product_id,event_type,event_time,message_time,received_at_ns,side,price_level,new_quantity,sequence_num
0,coinbase,SOL-USD,snapshot,2026-10-09T10:38:46.321411Z,2026-10-09T10:38:46.455198667Z,1791542326505635000,bid,109.6,63.13309409,2
1,coinbase,SOL-USD,snapshot,2026-10-09T10:38:46.321411Z,2026-10-09T10:38:46.455198667Z,1791542326505635000,bid,109.59,67.81322845,2
2,coinbase,SOL-USD,snapshot,2026-10-09T10:38:46.321411Z,2026-10-09T10:38:46.455198667Z,1791542326505635000,bid,109.58,82.64880602,2
3,coinbase,SOL-USD,snapshot,2026-10-09T10:38:46.321411Z,2026-10-09T10:38:46.455198667Z,1791542326505635000,bid,109.57,118.59579533,2
4,coinbase,SOL-USD,snapshot,2026-10-09T10:38:46.321411Z,2026-10-09T10:38:46.455198667Z,1791542326505635000,bid,109.56,179.33954127,2


In [3]:
output_dir = Path("../data/ticks")
output_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

if trades.empty:
    print("No trades arrived during the sample.")
else:
    trades = trades.drop_duplicates(["exchange", "product_id", "trade_id"])
    trades["exchange_time"] = pd.to_datetime(trades["exchange_time"], utc=True)
    trades_path = output_dir / f"coinbase_SOL-USD_trades_{stamp}.csv"
    trades.to_csv(trades_path, index=False)
    print(f"Saved {len(trades):,} unique trades to {trades_path.resolve()}")

if level2.empty:
    print("No Level 2 rows arrived during the sample.")
else:
    level2["event_time"] = pd.to_datetime(level2["event_time"], utc=True)
    level2["message_time"] = pd.to_datetime(level2["message_time"], utc=True)
    level2_path = output_dir / f"coinbase_SOL-USD_level2_{stamp}.csv"
    level2.to_csv(level2_path, index=False)
    print(f"Saved {len(level2):,} Level 2 rows to {level2_path.resolve()}")

display(trades.tail())
display(level2.tail())

Saved 14,625 unique trades to /Users/hankrugg/Fordham/Fall26/Crypto/trading/data/ticks/coinbase_SOL-USD_trades_20261009T133847Z.csv
Saved 942,749 Level 2 rows to /Users/hankrugg/Fordham/Fall26/Crypto/trading/data/ticks/coinbase_SOL-USD_level2_20261009T133847Z.csv


,exchange,product_id,trade_id,exchange_time,received_at_ns,price,size,maker_side,event_type,sequence_num
15101,coinbase,SOL-USD,357192942,2026-10-09 13:38:43.452216+00:00,1791553123541965000,109.42,0.03342424,SELL,update,156514
15102,coinbase,SOL-USD,357192941,2026-10-09 13:38:43.452216+00:00,1791553123541965000,109.42,0.011,SELL,update,156514
15103,coinbase,SOL-USD,357192943,2026-10-09 13:38:43.717716+00:00,1791553123742485000,109.42,4.56955571,SELL,update,156515
15104,coinbase,SOL-USD,357192944,2026-10-09 13:38:44.324084+00:00,1791553124346243000,109.41,9.23724163,BUY,update,156525
15105,coinbase,SOL-USD,357192945,2026-10-09 13:38:45.582021+00:00,1791553125741163000,109.43,1.55199591,BUY,update,156546


,exchange,product_id,event_type,event_time,message_time,received_at_ns,side,price_level,new_quantity,sequence_num
942744,coinbase,SOL-USD,update,2026-10-09 13:38:46.039655+00:00,2026-10-09 13:38:46.130264352+00:00,1791553126134019000,offer,109.47,143.2348257,156556
942745,coinbase,SOL-USD,update,2026-10-09 13:38:46.039655+00:00,2026-10-09 13:38:46.130264352+00:00,1791553126134019000,offer,109.56,249.84642309,156556
942746,coinbase,SOL-USD,update,2026-10-09 13:38:46.039655+00:00,2026-10-09 13:38:46.130264352+00:00,1791553126134019000,offer,109.57,233.36680392,156556
942747,coinbase,SOL-USD,update,2026-10-09 13:38:46.041858+00:00,2026-10-09 13:38:46.153634082+00:00,1791553126157225000,offer,109.58,370.0813873,156557
942748,coinbase,SOL-USD,update,2026-10-09 13:38:46.041858+00:00,2026-10-09 13:38:46.153634082+00:00,1791553126157225000,offer,109.59,935.94857244,156557


## Before running this continuously

A production collector still needs automatic reconnection, sequence-gap alerts, idempotent writes, daily file rotation, and REST reconciliation after disconnects. Reconstruct Level 2 by loading the latest `snapshot`, then applying each `update` in order. This feed contains aggregate quantities at price levels, not individual orders or queue positions.

In [4]:
trades.received_at_ns.nunique()

7289

In [8]:
level2.head()


,exchange,product_id,event_type,event_time,message_time,received_at_ns,side,price_level,new_quantity,sequence_num
0,coinbase,SOL-USD,snapshot,2026-10-09 10:38:46.321411+00:00,2026-10-09 10:38:46.455198667+00:00,1791542326505635000,bid,109.6,63.13309409,2
1,coinbase,SOL-USD,snapshot,2026-10-09 10:38:46.321411+00:00,2026-10-09 10:38:46.455198667+00:00,1791542326505635000,bid,109.59,67.81322845,2
2,coinbase,SOL-USD,snapshot,2026-10-09 10:38:46.321411+00:00,2026-10-09 10:38:46.455198667+00:00,1791542326505635000,bid,109.58,82.64880602,2
3,coinbase,SOL-USD,snapshot,2026-10-09 10:38:46.321411+00:00,2026-10-09 10:38:46.455198667+00:00,1791542326505635000,bid,109.57,118.59579533,2
4,coinbase,SOL-USD,snapshot,2026-10-09 10:38:46.321411+00:00,2026-10-09 10:38:46.455198667+00:00,1791542326505635000,bid,109.56,179.33954127,2


In [5]:
trades[trades['event_type'] == 'update']

,exchange,product_id,trade_id,exchange_time,received_at_ns,price,size,maker_side,event_type,sequence_num
100,coinbase,SOL-USD,357178421,2026-10-09 10:38:49.478435+00:00,1791542329568748000,109.62,2.65902226,SELL,update,55
101,coinbase,SOL-USD,357178422,2026-10-09 10:38:50.352697+00:00,1791542330370246000,109.62,0.10946907,SELL,update,67
102,coinbase,SOL-USD,357178423,2026-10-09 10:38:54.383706+00:00,1791542334472314000,109.63,0.04515599,SELL,update,121
103,coinbase,SOL-USD,357178448,2026-10-09 10:38:55.940019+00:00,1791542335970622000,109.57,42.19111559,BUY,update,139
104,coinbase,SOL-USD,357178447,2026-10-09 10:38:55.940019+00:00,1791542335970622000,109.57,13.90864048,BUY,update,139
...,...,...,...,...,...,...,...,...,...,...
15101,coinbase,SOL-USD,357192942,2026-10-09 13:38:43.452216+00:00,1791553123541965000,109.42,0.03342424,SELL,update,156514
15102,coinbase,SOL-USD,357192941,2026-10-09 13:38:43.452216+00:00,1791553123541965000,109.42,0.011,SELL,update,156514
15103,coinbase,SOL-USD,357192943,2026-10-09 13:38:43.717716+00:00,1791553123742485000,109.42,4.56955571,SELL,update,156515
15104,coinbase,SOL-USD,357192944,2026-10-09 13:38:44.324084+00:00,1791553124346243000,109.41,9.23724163,BUY,update,156525


In [6]:
level2[level2['event_type'] == 'update']

,exchange,product_id,event_type,event_time,message_time,received_at_ns,side,price_level,new_quantity,sequence_num
13118,coinbase,SOL-USD,update,2026-10-09 10:38:46.386537+00:00,2026-10-09 10:38:46.462389348+00:00,1791542326512229000,bid,109.49,534.39080789,3
13119,coinbase,SOL-USD,update,2026-10-09 10:38:46.386537+00:00,2026-10-09 10:38:46.462389348+00:00,1791542326512229000,bid,109.33,90.8171047,3
13120,coinbase,SOL-USD,update,2026-10-09 10:38:46.386537+00:00,2026-10-09 10:38:46.462389348+00:00,1791542326512229000,bid,95.91,7.85720835,3
13121,coinbase,SOL-USD,update,2026-10-09 10:38:46.386537+00:00,2026-10-09 10:38:46.462389348+00:00,1791542326512229000,offer,109.62,46.17129703,3
13122,coinbase,SOL-USD,update,2026-10-09 10:38:46.386537+00:00,2026-10-09 10:38:46.462389348+00:00,1791542326512229000,offer,109.69,320.20710197,3
...,...,...,...,...,...,...,...,...,...,...
942744,coinbase,SOL-USD,update,2026-10-09 13:38:46.039655+00:00,2026-10-09 13:38:46.130264352+00:00,1791553126134019000,offer,109.47,143.2348257,156556
942745,coinbase,SOL-USD,update,2026-10-09 13:38:46.039655+00:00,2026-10-09 13:38:46.130264352+00:00,1791553126134019000,offer,109.56,249.84642309,156556
942746,coinbase,SOL-USD,update,2026-10-09 13:38:46.039655+00:00,2026-10-09 13:38:46.130264352+00:00,1791553126134019000,offer,109.57,233.36680392,156556
942747,coinbase,SOL-USD,update,2026-10-09 13:38:46.041858+00:00,2026-10-09 13:38:46.153634082+00:00,1791553126157225000,offer,109.58,370.0813873,156557
